In [1]:
library(xgboost)
library(lime)

# Load necessary libraries
library(caret)
library(pROC)  # For AUC-ROC calculation


Loading required package: ggplot2

Loading required package: lattice

Type 'citation("pROC")' for a citation.


Attaching package: ‘pROC’


The following objects are masked from ‘package:stats’:

    cov, smooth, var




In [62]:
# Custom xgbFuncs tailored for RFE with XGBoost
xgbFuncs <- list(
  fit = function(x, y, first, last, ...) {
    # Define parameters for xgboost (customize as required)
    # Ensure that ... contains any other arguments (e.g., the number of rounds nrounds, etc.)
    params <- list(objective = "binary:logistic", ...)
    xgboost_model <- xgb.train(params=params,
      data = xgb.DMatrix(data = as.matrix(x[first:last, , drop = FALSE]), label = y[first:last]),
      verbose = 0, # silent = TRUE to suppress iteration printing
      ...)
    return(xgboost_model)
  },
  predict = function(modelFit, newdata, preProc = NULL, submodels = NULL) {
    # Ensure newdata is a matrix
    newdata <- as.matrix(newdata)
    # Make predictions using the xgboost model
    preds <- predict(modelFit, xgb.DMatrix(newdata))
    if (ncol(preds) == 2) {
      preds <- preds[,1]
    } 
    return(preds)
  },
  rank = function(modelFit, x, y) {
    # Extract feature importance from the xgboost model
    imp <- xgb.importance(feature_names = colnames(x), model = modelFit)
    importance <- imp$Gain
    names(importance) <- imp$Feature
    return(importance)
  }
)

xgbFuncs$summary <- function(data, lev = NULL, model = NULL) {
  # The observed values 'data$obs' should be a factor with levels: 'lev[1]' and 'lev[2]'
  # The predicted probabilities 'data$pred' is the column that contains the model's class probabilities 
  # for the class of interest (class 1 in the case of binary classification)

  if (is.numeric(data$obs) && length(unique(data$obs)) > 1) {
    rocAUC <- pROC::roc(response = as.numeric(data$obs),
                        predictor = as.numeric(data$pred),
                        levels = rev(lev))
    out <- c(ROC = rocAUC$auc)
  } else {
    out <- c(ROC = NA)
  }
  out
}

xgbFuncs$fit <- function(x, y, first, last, ...) {
  # Access arguments passed in '...' and specify defaults for important arguments if not provided
  args <- list(...)
  if(!("nrounds" %in% names(args))) {
    args$nrounds <- 100  # Setting a default; it should be tuned according to your dataset
  }
  params <- list(objective = "binary:logistic", ...)
  params <- c(params, args)  # Combine params and additional arguments

  xgb.train(params=params,
            data = xgb.DMatrix(data = as.matrix(x[first:last, , drop = FALSE]), label = y[first:last]),
            verbose = 0, # silent = TRUE to suppress iteration printing
            ...)  # Pass additional arguments to xgb.train
}

In [63]:
## Read previously processed data 
scaled_df = read.csv('../data/merged_dataset.csv')
meta = read.csv('../data/merged_dataset_meta.csv')
response_name = 'cluster'

## Subset only the patients from the BECAME cohort
became_data = scaled_df[rownames(subset(meta,cohort=='BECAME' & group=='HFpEF')),]

In [64]:
## Subset the metadata 
became_meta = meta[rownames(became_data),]

## Log transform the intensities 
#became_data = log(became_num[rownames(became_data),])
became_data$response_var = as.factor(as.numeric(became_meta[rownames(became_data),response_name]=='B1'))

## Create train/test indices 
index <- createDataPartition(became_data$response_var, p = 0.7, list = FALSE)

## Define training data 
X = became_data[index,]
y = as.factor(as.character(became_data[index,'response_var']))

In [65]:
dtrain <- xgb.DMatrix(data = as.matrix(X[,-ncol(X)]), label = y)

# Set parameters
params <- list(
  objective = "binary:logistic",
  eval_metric = "logloss"
)


In [66]:
y

[1] 0 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 1 0 0 0 1 0 0 1 1 0 1 0
[39] 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 1 0 1 0
Levels: 0 1

In [67]:
# Use RFE to find the best subset of features
controlXGB <- rfeControl(functions = xgbFuncs,
                         method = "repeatedcv",
                         repeats = 5,
                         verbose = FALSE)
rfe_results <- rfe(
  x = as.matrix(X[,-ncol(X)]),    # Your feature set
  y = y,               # Your binary response variable
  sizes = c(1:(ncol(X)-1)), # Range of feature subset sizes to evaluate
  rfeControl = controlXGB,
  # Set any additional arguments that are required by xgb.train, such as nrounds
  nrounds = 100  # Number of boosting rounds to use in xgboost, adjust as necessary
)
# Print RFE results
print(results)

# Get the selected subset of features
optimal_features <- predictors(results)
print(optimal_features)

Warning message in rfe.default(x = as.matrix(X[, -ncol(X)]), y = y, sizes = c(1:(ncol(X) - :
“Metric 'Accuracy' is not created by the summary function; 'ROC' will be used instead”
Warning message in check.booster.params(params, ...):
“The following parameters were provided multiple times:
	nrounds
  Only the last value for each of them will be used.
”


[19:44:19] WARNING: src/learner.cc:767: 
Parameters: { "nrounds" } are not used.



Warning message in check.booster.params(params, ...):
“The following parameters were provided multiple times:
	nrounds
  Only the last value for each of them will be used.
”


[19:44:19] WARNING: src/learner.cc:767: 
Parameters: { "nrounds" } are not used.



Warning message in check.booster.params(params, ...):
“The following parameters were provided multiple times:
	nrounds
  Only the last value for each of them will be used.
”


...

Warning message in check.booster.params(params, ...):
“The following parameters were provided multiple times:
	nrounds
  Only the last value for each of them will be used.
”


[19:44:20] WARNING: src/learner.cc:767: 
Parameters: { "nrounds" } are not used.



Warning message in check.booster.params(params, ...):
“The following parameters were provided multiple times:
	nrounds
  Only the last value for each of them will be used.
”


[19:44:20] WARNING: src/learner.cc:767: 
Parameters: { "nrounds" } are not used.



ERROR: Error in {: task 1 failed - "argument is of length zero"


In [16]:


# Train the XGBoost model
model <- xgboost(data = dtrain, 
                 params = params, 
                 nrounds = 50,
                 verbose=0)

X_test <- became_data[-index,grepl('^X',colnames(became_data))]
y_test <- factor(as.numeric(as.character(became_data[-index,]$response_var)))

# Predictions on the test set
predictions <- predict(model, as.matrix(X_test))

# Convert predicted probabilities to class labels (binary classification)
predicted_labels <- ifelse(predictions > 0.5, 1, 0)
table(predicted_labels,y_test)

importance <- data.frame(xgb.importance(model = model))
rownames(importance) = importance[,1]

                y_test
predicted_labels  0  1
               0 23  3
               1  0  4